# NLP Lab Test — Short Notebook (Tasks A–E)
**Exam revision version:** short, separate blocks, easy to run one task at a time.

Run the setup cell first. Then run each task's cells. If a library/model is unavailable, skip that optional section and practise the core code.

**Install once in terminal:** `pip install nltk numpy scikit-learn spacy transformers torch gensim`  
**spaCy model:** `python -m spacy download en_core_web_sm`

## Setup — run once

In [2]:
import torch
print("Torch:", torch.__version__)

import spacy
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded!")

Torch: 2.14.1+cpu
spaCy model loaded!


In [3]:
import re, math, random
from collections import Counter
random.seed(42)

import nltk
for r in ["punkt", "punkt_tab", "wordnet", "omw-1.4",
          "averaged_perceptron_tagger_eng"]:
    nltk.download(r, quiet=True)

def tokens(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace(":)", " <happy> ").replace(":(", " <sad> ")
    text = text.replace("can't", "cannot").replace("n't", " not")
    return re.findall(r"<happy>|<sad>|[a-z]+|\d+|[^\w\s]", text)

text = "I can't believe this movie :) https://test.com"
print(tokens(text))

['i', 'cannot', 'believe', 'this', 'movie', '<happy>']


## A1 — Porter stemming vs WordNet lemmatization

In [4]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

words = ["playing", "studies", "cars", "better"]
ps, lem = PorterStemmer(), WordNetLemmatizer()
print("Stemming:", [ps.stem(w) for w in words])
print("Lemmatizing:", [lem.lemmatize(w) for w in words])

Stemming: ['play', 'studi', 'car', 'better']
Lemmatizing: ['playing', 'study', 'car', 'better']


## A2 — Bigram language model, add-one smoothing, probability, perplexity, sampling

In [5]:
train = [tokens(s) for s in [
    "i love nlp", "i love language", "language is useful"
]]
unigram, bigram = Counter(), Counter()
vocab = {"<s>", "</s>", "<unk>"}
for sent in train:
    vocab.update(sent)
    seq = ["<s>"] + sent + ["</s>"]
    unigram.update(seq[:-1])
    bigram.update(zip(seq[:-1], seq[1:]))
V = len(vocab)

def prob(word, prev):
    return (bigram[(prev, word)] + 1) / (unigram[prev] + V)

def sentence_prob(words):
    seq = ["<s>"] + words + ["</s>"]
    p = 1
    for prev, word in zip(seq, seq[1:]):
        p *= prob(word if word in vocab else "<unk>", prev)
    return p

def perplexity(words):
    seq = ["<s>"] + words + ["</s>"]
    logp = sum(math.log(prob(w if w in vocab else "<unk>", p))
                for p, w in zip(seq, seq[1:]))
    return math.exp(-logp / (len(seq)-1))

print("Probability:", sentence_prob(tokens("i love nlp")))
print("Perplexity:", perplexity(tokens("i love nlp")))

def sample(seed=42, length=10):
    rng, out, prev = random.Random(seed), [], "<s>"
    for _ in range(length):
        choices = sorted(vocab)
        w = rng.choices(choices, [prob(x, prev) for x in choices])[0]
        if w == "</s>": break
        if w != "<s>": out.append(w)
        prev = w
    return " ".join(out)
print("Sample:", sample())

Probability: 0.0024793388429752063
Perplexity: 4.4814239301650645
Sample: language


## B1 — POS tagging + open-class and closed-class words

In [7]:
import nltk
from nltk import pos_tag, word_tokenize

sentence = "The clever student reads books quickly."
tagged = pos_tag(word_tokenize(sentence))
print(tagged)

open_tags = {"NN","NNS","NNP","NNPS","VB","VBD","VBG","VBN","VBP","VBZ","JJ","JJR","JJS","RB","RBR","RBS"}
closed_tags = {"DT","IN","PRP","PRP$","CC","TO","MD"}
print("Open:", [(w,t) for w,t in tagged if t in open_tags])
print("Closed:", [(w,t) for w,t in tagged if t in closed_tags])

[('The', 'DT'), ('clever', 'JJ'), ('student', 'NN'), ('reads', 'NNS'), ('books', 'NNS'), ('quickly', 'RB'), ('.', '.')]
Open: [('clever', 'JJ'), ('student', 'NN'), ('reads', 'NNS'), ('books', 'NNS'), ('quickly', 'RB')]
Closed: [('The', 'DT')]


## B2 — CFG + ChartParser: PP attachment ambiguity

In [7]:

from nltk import CFG
from nltk.parse import ChartParser

grammar = CFG.fromstring('''
S -> NP VP
NP -> Det N | NP PP
VP -> V NP | VP PP
PP -> P NP
Det -> 'the'
N -> 'man' | 'telescope'
V -> 'saw'
P -> 'with'
''')

parser = ChartParser(grammar)

sentence = "the man saw the man with the telescope".split()

for tree in parser.parse(sentence):
    print(tree)
    print()


(S
  (NP (Det the) (N man))
  (VP
    (VP (V saw) (NP (Det the) (N man)))
    (PP (P with) (NP (Det the) (N telescope)))))

(S
  (NP (Det the) (N man))
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det the) (N telescope))))))



## B3 — spaCy dependencies + aspect/opinion pairs

In [8]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("The screen is bright. The camera is excellent.")
for t in doc:
    print(t.text, t.pos_, t.dep_, "head=", t.head.text)

pairs = set()
for t in doc:
    if t.dep_ == "amod":
        pairs.add((t.head.text, t.text))  # aspect, opinion
    if t.dep_ in {"acomp", "attr"}:
        for child in t.head.children:
            if child.dep_ == "nsubj":
                pairs.add((child.text, t.text))
print("Aspect-opinion:", pairs)

The DET det head= screen
screen NOUN nsubj head= is
is AUX ROOT head= is
bright ADJ acomp head= is
. PUNCT punct head= is
The DET det head= camera
camera NOUN nsubj head= is
is AUX ROOT head= is
excellent ADJ acomp head= is
. PUNCT punct head= is
Aspect-opinion: {('screen', 'bright'), ('camera', 'excellent')}


## C1 — WordNet synsets, Wu–Palmer, Lesk

In [9]:
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

print("Synsets:", wn.synsets("bank")[:5])
print("Wu-Palmer dog/cat:",
      wn.synset("dog.n.01").wup_similarity(wn.synset("cat.n.01")))
context = nltk.word_tokenize("The fisherman sat by the bank of the river")
print("Lesk sense:", lesk(context, "bank", "n"))

Synsets: [Synset('bank.n.01'), Synset('depository_financial_institution.n.01'), Synset('bank.n.03'), Synset('bank.n.04'), Synset('bank.n.05')]
Wu-Palmer dog/cat: 0.8571428571428571
Lesk sense: Synset('bank.n.01')


## C2 — TF-IDF, cosine similarity, top-k retrieval

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = ["cats are animals", "language models predict words",
        "NLP processes human language"]
query = "language models"

vec = TfidfVectorizer()
X = vec.fit_transform(docs + [query])

scores = cosine_similarity(X[-1], X[:-1]).ravel()
for i in scores.argsort()[::-1][:2]:
    print(round(scores[i], 3), docs[i])

0.583 language models predict words
0.218 NLP processes human language


## C3 — PPMI window ±2 + Word2Vec skip-gram

In [11]:
corpus = [tokens("cats chase small animals"), tokens("dogs chase cats")]
uni, pairs = Counter(), Counter()
total = 0
for sent in corpus:
    uni.update(sent)
    for i, w in enumerate(sent):
        for j in range(max(0,i-2), min(len(sent),i+3)):
            if i != j:
                pairs[(w,sent[j])] += 1
                total += 1
for (w,c), n in pairs.items():
    pmi = math.log2(n * total / (uni[w] * uni[c]))
    print(w, c, "PPMI=", round(max(0,pmi), 3))

# Optional Word2Vec
from gensim.models import Word2Vec
model = Word2Vec(corpus, vector_size=20, window=2, min_count=1,
                 sg=1, seed=42, workers=1)
print(model.wv.most_similar("cats", topn=2))

cats chase PPMI= 3.0
cats small PPMI= 3.0
chase cats PPMI= 3.0
chase small PPMI= 3.0
chase animals PPMI= 3.0
small cats PPMI= 3.0
small chase PPMI= 3.0
small animals PPMI= 4.0
animals chase PPMI= 3.0
animals small PPMI= 4.0
dogs chase PPMI= 3.0
dogs cats PPMI= 3.0
chase dogs PPMI= 3.0
cats dogs PPMI= 3.0
[('small', 0.13939636945724487), ('dogs', 0.07304178178310394)]


## D — Coreference: short rule-based mention-pair scoring

In [12]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("John met Mary. He thanked her. Mary smiled.")
mentions = []
gender = {"john":"M", "he":"M", "him":"M", "mary":"F", "she":"F", "her":"F"}
for ent in doc.ents:
    if ent.label_ == "PERSON":
        mentions.append((ent.text, ent.start, gender.get(ent.text.lower(), "?"), "name"))
for t in doc:
    if t.pos_ == "PRON":
        mentions.append((t.text, t.i, gender.get(t.text.lower(), "?"), "pronoun"))
mentions.sort(key=lambda x: x[1])
print("Mentions:", mentions)

def score(antecedent, mention):
    # tuple: text, token_position, gender, kind
    if antecedent[1] >= mention[1]: return -999
    if antecedent[2] != "?" and mention[2] != "?" and antecedent[2] != mention[2]:
        return -999
    return (3 if antecedent[2] == mention[2] else 0) - (mention[1]-antecedent[1]) * 0.1

for m in mentions:
    if m[3] == "pronoun":
        prior = [a for a in mentions if a[1] < m[1]]
        if prior:
            best = max(prior, key=lambda a: score(a,m))
            print(m[0], "->", best[0])

Mentions: [('John', 0, 'M', 'name'), ('Mary', 2, 'F', 'name'), ('He', 4, 'M', 'pronoun'), ('her', 6, 'F', 'pronoun'), ('Mary', 8, 'F', 'name')]
He -> John
her -> Mary


In [16]:
import os
os.environ["USE_TF"] = "0"

from transformers import pipeline
print("Transformers pipeline imported successfully!")

Transformers pipeline imported successfully!


In [1]:

import os
print(os.path.expanduser("~/.cache/huggingface/hub"))


C:\Users\amsav/.cache/huggingface/hub


In [4]:
import requests

url = "https://huggingface.co/google-bert/bert-base-uncased/resolve/main/config.json"

try:
    r = requests.get(url, timeout=20)
    print("Status:", r.status_code)
except Exception as e:
    print(type(e).__name__, e)

Status: 200


## E1 — Fill-mask + sentiment pipelines

In [5]:
import os
os.environ["USE_TF"] = "0"

from transformers import pipeline

fill = pipeline(
    "fill-mask",
    model="google-bert/bert-base-uncased",
    framework="pt"
)

print(fill("Natural language processing is [MASK].", top_k=3))


ValueError: Could not load model google-bert/bert-base-uncased with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForMaskedLM'>, <class 'transformers.models.bert.modeling_bert.BertForMaskedLM'>). See the original errors:

while loading with AutoModelForMaskedLM, an error is thrown:
Traceback (most recent call last):
  File "C:\Users\amsav\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py", line 788, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\amsav\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py", line 488, in _make_request
    raise new_e
  File "C:\Users\amsav\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py", line 464, in _make_request
    self._validate_conn(conn)
  File "C:\Users\amsav\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py", line 1106, in 

## E2 — Fine-tune BERT on 9 samples + accuracy/F1 + new prediction

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

texts = ["Amazing movie", "I loved it", "Excellent acting", "Very good",
         "Wonderful story", "Terrible movie", "I hated it", "Very boring",
         "Awful acting"]
labels = [1,1,1,1,1,0,0,0,0]  # 1=positive, 0=negative

# Tiny demonstration split: 6 train, 3 test
train_x, train_y = texts[:6], labels[:6]
test_x, test_y = texts[6:], labels[6:]

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)

enc = tok(train_x, padding=True, truncation=True, return_tensors="pt").to(device)
y = torch.tensor(train_y).to(device)
model.train()
opt.zero_grad()
loss = model(**enc, labels=y).loss
loss.backward()
opt.step()
print("Training loss:", loss.item())

model.eval()
test_enc = tok(test_x, padding=True, truncation=True, return_tensors="pt").to(device)
with torch.no_grad():
    pred = model(**test_enc).logits.argmax(dim=1).cpu().tolist()
print("Actual:", test_y, "Predicted:", pred)
print("Accuracy:", accuracy_score(test_y, pred))
print("F1:", f1_score(test_y, pred, zero_division=0))

new_text = "The film was engaging and beautifully acted."
new_enc = tok(new_text, return_tensors="pt", truncation=True).to(device)
with torch.no_grad():
    label = model(**new_enc).logits.argmax(dim=1).item()
print("New prediction:", "positive" if label == 1 else "negative")

SafetensorError: Error while deserializing header: incomplete metadata, file not fully covered

## Last-minute memory sheet

- **Preprocessing:** lowercase → remove URLs → expand contractions → map emoticons → regex tokens.
- **Stemming:** chops word endings; **lemmatization:** returns a dictionary form.
- **Add-one bigram:** `(count(prev, word) + 1) / (count(prev) + vocabulary_size)`.
- **Perplexity:** `exp(-average log probability)`; lower is usually better.
- **POS:** open-class = nouns/verbs/adjectives/adverbs; closed-class = determiners/prepositions/pronouns/conjunctions.
- **CFG ambiguity:** PP can attach to NP or VP.
- **Dependencies:** `nsubj` = subject; `amod` = adjective modifier; `acomp` = adjectival complement.
- **WordNet:** synsets; **Wu–Palmer:** taxonomy similarity; **Lesk:** context-based word-sense selection.
- **TF-IDF/cosine:** rank documents by similarity to query.
- **PPMI:** `max(0, log2(P(w,c)/(P(w)P(c))))`; Word2Vec `sg=1` means skip-gram.
- **Coreference:** link mentions using agreement and distance; this notebook uses simple rules.
- **BERT:** fill-mask predicts `[MASK]`; sentiment predicts polarity; fine-tuning adjusts weights for labels.
- **Accuracy:** correct/total. **F1:** harmonic mean of precision and recall.

**Important:** Task E downloads pretrained weights and can take time. The nine-example model is only a lab demonstration, not a reliable classifier.